# framediff: the decomposer on real transitions

`Decomposer().decompose(a, b)` searches for a short edit script from frame `a` to frame `b` and returns it as a `Transform`: `NoOp` when the frames are equal, otherwise a `Composite` of fills, recolours, block copies, lines, pastes and single cells. `explain(a, b, name_a, name_b)` is the same as a block of code.

The search is greedy on one objective, the change in the script's total cost with the single-cell fallback priced in: `cost − 4·(cells made right − cells made wrong)`. Each step takes the candidate with the lowest score and stops when none scores below zero; whatever is still wrong is then set cell by cell. Candidates come from gain maps and their maximum-sum rectangle (fills, recolours, pastes, and block copies at shifts shortlisted by FFT correlation, for all eight mirrored-and-turned variants of `a`), and from the connected components of wrong cells (lines). A paste is carved before it is proposed: the cheapest other transforms inside its rectangle are tried, followed by pastes of what they leave, since a paste's per-cell price otherwise absorbs whatever lies beside the content it is for.

This notebook draws transitions from the recorded `seal_wm_full` run: a game, then a level of it, then a move of that level, each uniformly, so long levels do not dominate. For every transition the decomposer runs against two baselines, the frame before the move and the frame the level opened with, and the cheaper block is shown. Set the seed and the sample size below.

In [ ]:
SEED = 0
N_SAMPLES = 12
POOL = 200  # transitions the worst cases and the summary are drawn from
N_WORST = 6

In [ ]:
import sys

sys.path.insert(0, ".")

from run_transitions import DEFAULT_RUN, load_states, sample_transitions, transition

from framediff.diagnostics import show_transition

states = load_states(DEFAULT_RUN)
samples = sample_transitions(states, SEED, N_SAMPLES)
sum(len(game) for game in states), len(states)

## The samples

Each panel: the two baselines and the current frame, with the cells that differ between the chosen baseline and the current frame outlined; a table of both searches (cells that differ, bit cost, edits, milliseconds); and the chosen block, written against the names `previous`, `level_start` and `current`. The first move of a level has the same frame as both baselines.

In [ ]:
from IPython.display import display

for sample in samples:
    display(
        show_transition(
            sample.current,
            {"previous": sample.previous, "level_start": sample.level_start},
            title=sample.title,
        )
    )

## A swap, read the other way

The costs set the order the search reaches for things in. Here the player moved two tiles down inside a column of tiles and turned to face down. A person writes the player's move; at the same cost the search may write the white tile's move the other way, leaving blue behind, then the eye. With the turned copy at 14 this used to come out as the horizontal strip that held the player turned clockwise into the vertical one; at 18 the move wins. The search takes the cheapest reading the table allows, and when that reading is the wrong one it is the table that changes, not the search.

In [ ]:
turn = transition(states, game_index=72, level=1, move=10)
display(show_transition(turn.current, {"previous": turn.previous}, title=turn.title))

## A larger pool

The same search over more transitions, drawn with the next seed. Only the cheaper of the two baselines counts for a transition; the table summarises that choice, then how long each search took and what it reached for.

In [ ]:
import collections

import numpy as np

from framediff import Composite
from framediff.diagnostics import cheapest, explain_all

pool = sample_transitions(states, SEED + 1, POOL)
explained = [
    explain_all(sample.current, {"previous": sample.previous, "level_start": sample.level_start}) for sample in pool
]
chosen = [cheapest(explanations) for explanations in explained]

costs = np.array([explanation.cost for explanation in chosen])
print(
    f"cheapest baseline per transition: cost median {np.median(costs):.0f}  p90 {np.percentile(costs, 90):.0f}  max {costs.max():.0f}"
)
print(f"chosen: {collections.Counter(explanation.name for explanation in chosen)}")
for name in ("previous", "level_start"):
    ms = (
        np.array(
            [
                explanation.seconds
                for explanations in explained
                for explanation in explanations
                if explanation.name == name
            ]
        )
        * 1000
    )
    print(f"{name:12s} ms median {np.median(ms):6.1f}  p90 {np.percentile(ms, 90):6.1f}  max {ms.max():6.1f}")
kinds = collections.Counter()
for explanation in chosen:
    if isinstance(explanation.transform, Composite):
        kinds.update(type(part).__name__ for part in explanation.transform.transforms)
kinds

## The worst of the pool

The transitions whose cheaper baseline still costs the most: what the search cannot describe briefly.

In [ ]:
worst = sorted(range(len(pool)), key=lambda i: -chosen[i].cost)[:N_WORST]
for i in worst:
    display(
        show_transition(
            pool[i].current,
            {"previous": pool[i].previous, "level_start": pool[i].level_start},
            title=pool[i].title,
        )
    )